In [1]:
# Required packages
import os
import subprocess
import datetime

In [ ]:
# The code below assumes that all required files (ilastik model for cell classification, maxZ of cellbody + nuclues, and segmentation maps of the cellbody and nuclues) are in the same folder
# Specify the path to this folder below
main_folder = str('PATH TO FOLDER CONTAINING RGB IMAGES AND ILASTIK MODEL')
# Replicate - used to add experimental repeats
replicate = str('Experiment1')

# Specify the file name of the ilastik model to be executed (relies on it being the main folder);
project_obj = main_folder + str('ILASTIK MODEL')
# Specify output folderf
output_folder_obj = main_folder + replicate + str('OUTPUT DIRECTORY')

# Specify the path to the folder with RGB maxZ .tifs
image_folder = main_folder + replicate + str('PATH TO FOLDER WITH MAXZ RGBS')
# Specify the path to the folder with CellPose segmentation masks (just the cellbody)
segmentation_folder = main_folder + replicate + str('PATH TO FOLDER WITH CELLPOSE SEGMENTATION MAPS OF CELLBODY')

# Specify the location of the Ilastik software
ilastik_location = 'PATH TO ILASTIK .EXE'

In [3]:
type(os.listdir(image_folder))

list

In [4]:
#Function that all files in the specified directory (can filter according to whether a specific word, e.g. an extension is present) 
#and generates a list of the absolute paths to the files in the directory, converting the windows path notation to the linux format
def FilePathList_linux(FOLDER, file_extension):
    input_files = os.listdir(FOLDER)
    input_files.sort()
    input_path = []
    for i in input_files:
        if(file_extension in i):           
            path_windows = os.path.join(FOLDER,i)
            path_linux = path_windows.replace("\\","/")
            input_path.append(path_linux)
    return input_path

In [5]:
image_paths = FilePathList_linux(image_folder, '.tif')
segmentation_paths = FilePathList_linux(segmentation_folder, '.png')

print(image_paths[514].rpartition("/")[2], segmentation_paths[514].rpartition("/")[2])

r03c01f27p04_ch1_ch2_RGB.tif r03c01f27p04_ch1_ch2_RGB_cp_masks.png


In [6]:
#Tests whether the lists of corrected channel images match the segmentation images (in terms of FOV + slice information, which are the first 12 characters in the file name)
def image_and_segmentation_matched(images, segmentations):
    if(len(images) == len(segmentations)):
        for i in range(0, len(images)):
            image_file = images[i].rpartition("/")[2]
            segmentation_file = segmentations[i].rpartition("/")[2] 
            if(image_file[0:12] == segmentation_file[0:12]):
                print('No. ', i+1, ': Image and segmentation correctly matched')
            else:
                print('No. ', i+1 , ': No match for image ',image_file[0:12])
                raise RuntimeError('IMPERFECT MATCH BETWEEN IMAGE and SEGMENTATION') from None
        print("All images and segmentations correctly matched")
    else:
        raise RuntimeError('Length of images and segmentations are different (different no. of files in the lists)') from None

In [7]:
image_and_segmentation_matched(image_paths, segmentation_paths)

No.  1 : Image and segmentation correctly matched
No.  2 : Image and segmentation correctly matched
No.  3 : Image and segmentation correctly matched
No.  4 : Image and segmentation correctly matched
No.  5 : Image and segmentation correctly matched
No.  6 : Image and segmentation correctly matched
No.  7 : Image and segmentation correctly matched
No.  8 : Image and segmentation correctly matched
No.  9 : Image and segmentation correctly matched
No.  10 : Image and segmentation correctly matched
No.  11 : Image and segmentation correctly matched
No.  12 : Image and segmentation correctly matched
No.  13 : Image and segmentation correctly matched
No.  14 : Image and segmentation correctly matched
No.  15 : Image and segmentation correctly matched
No.  16 : Image and segmentation correctly matched
No.  17 : Image and segmentation correctly matched
No.  18 : Image and segmentation correctly matched
No.  19 : Image and segmentation correctly matched
No.  20 : Image and segmentation correct

In [ ]:
#Function that runs ilastik object classification module where the accepted inputs are: 
    #absolute path to project
    #list of absolute paths to images
    #list of absolute paths to segmentations
    #absolute path to output folder
    #Suffix to put on generated output file. The first 12 characters of the file is the FOV + slice values. Note, include file extension in the suffix, e.g. .png.
def Ilastik_ObjectClassification(project, images, segmentation, output_folder,output_suffix):
    ilastik_path = ilastik_location
    os.chdir(ilastik_path) #Sets working directory to Ilastik folder as running ilastik otherwise doesn't work
    begin_ilastik_time = datetime.datetime.now() #Begin_time variable is used to define a starting timepoint for when the function is executed (used to measure computing time)
    for i in range(0, len(images)): #For loop defined using range (so iterates through numbers from 0 to length)
        begin_image_time = datetime.datetime.now()
        output_file = os.path.join(output_folder+'/',images[i].rpartition("/")[2][0:12])+output_suffix
        subprocess.run('./run_ilastik.sh --headless --project="'+project+'" \
        --raw_data="'+images[i]+'" --segmentation_image="'+segmentation[i]+'" \
        --export_source="Object Predictions" --output_format=png --output_filename_format="'+output_file+'"'\
                       , shell=True, capture_output = True, text=True)
        print('Image processing time for iteration:',i,'was:', datetime.datetime.now()-begin_image_time)
    print('Total processing time was:', datetime.datetime.now()-begin_ilastik_time)

In [ ]:
Ilastik_ObjectClassification(project_obj, image_paths, segmentation_paths, output_folder_obj, 'OBJ_PRED.png')

Image processing time for iteration: 0 was: 0:00:24.443501
Image processing time for iteration: 1 was: 0:00:23.917444
Image processing time for iteration: 2 was: 0:00:25.401055
Image processing time for iteration: 3 was: 0:00:25.435751
